# NumCompute quickstart

End-to-end walk through the package: CSV in, preprocessing, statistics, sort/rank, metrics, gradients, pipeline and a benchmark.

In [ ]:
import numpy as np

from numcompute import preprocessing as pp
from numcompute import stats
from numcompute.utils import iter_batches

np.set_printoptions(precision=3, suppress=True)

## 1. Read CSV (`io.py`)

TODO (Jose): replace `np.genfromtxt` with `io.read_csv` once it lands. `sample.csv` has blank cells, which become NaN.

In [ ]:
X_num = np.genfromtxt('sample.csv', delimiter=',', skip_header=1, usecols=(0, 1, 2))          # age, income, height; blanks -> NaN
city = np.genfromtxt('sample.csv', delimiter=',', skip_header=1, usecols=3, dtype=str).reshape(-1, 1)
print(X_num)
print(city.ravel())

## 2. Preprocess (`preprocessing.py`)

Impute missing numbers, then scale. Fit on train, apply to new rows with the same statistics.

In [ ]:
imp = pp.SimpleImputer(strategy='median')
X_full = imp.fit_transform(X_num)
print('fill values:', imp.statistics_)

scaler = pp.StandardScaler().fit(X_full)
Z = scaler.transform(X_full)
print('mean after scaling:', Z.mean(axis=0), ' std:', Z.std(axis=0))

print(pp.MinMaxScaler((0, 1)).fit_transform(X_full)[:3])

In [ ]:
enc = pp.OneHotEncoder(handle_unknown='ignore').fit(city)
print('categories:', enc.categories_[0])
print(enc.transform(city))          # blank city -> all zeros
print(enc.transform([['Perth']]))    # unseen city, ignored

In [ ]:
# scaler fitted on 3 features rejects a 2-feature matrix with a clear message
try:
    scaler.transform(X_full[:, :2])
except ValueError as e:
    print('ValueError:', e)

## 3. Descriptive statistics (`stats.py`)

`axis=0` summarises each column. `skip_nan=True` ignores NaN; by default NaN propagates like NumPy.

In [ ]:
print('mean   ', stats.mean(X_num, axis=0))
print('mean   ', stats.mean(X_num, axis=0, skip_nan=True), '(skip NaN)')
print('median ', stats.median(X_num, axis=0, skip_nan=True))
print('std    ', stats.std(X_num, axis=0, ddof=1, skip_nan=True))
print('min/max', stats.min(X_num, axis=0, skip_nan=True), stats.max(X_num, axis=0, skip_nan=True))

In [ ]:
print(stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, skip_nan=True))   # shape (3, n_features)
counts, edges = stats.histogram(X_num[:, 1], bins=4)
print('income histogram:', counts, edges)

In [ ]:
# streaming stats: same numbers as the batch version, O(d) memory
rs = stats.RunningStats(skip_nan=True)
for batch in iter_batches(X_num, 3):
    rs.update(batch)
print('rows seen:', rs.n)
print('running mean:', rs.mean)
print('running std :', rs.std(ddof=1))

## 4. Sort, search, top-k and rank (`sort_search.py`, `rank.py`)

TODO (Faris): top-k on income, quickselect, binary search, rank with ties, percentile.

In [ ]:
# TODO (Faris)

## 5. Metrics (`metrics.py`)

TODO (Jose): accuracy / precision / recall / F1, confusion matrix, mse.

In [ ]:
# TODO (Jose)

## 6. Gradients (`optim.py`)

TODO (Shuhao): finite-difference `grad` and `jacobian` on a scalar and a vector function.

In [ ]:
# TODO (Shuhao)

## 7. Pipeline (`pipeline.py`)

TODO (Shuhao): `Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())]).fit_transform(X_num)`.

In [ ]:
# TODO (Shuhao)

## 8. Benchmark (`benchmarking.py`)

TODO (Jose): vectorised vs loop timings; paste the table into README.

In [ ]:
# TODO (Jose)